In [1]:
import pandas as pd
import numpy as np
import os

In [2]:
pd.set_option('display.max_columns', 50)

In [3]:
# add your local pathname here

df = pd.read_csv('/Users/jamesemcnally/Dropbox/DSBC Student Risk Factors Datasets/merged_streaming_data.csv')
df.head()

,id_student,code_module,code_presentation,date,forumng,homepage,oucontent,subpage,url,resource,glossary,dataplus,oucollaborate,quiz,ouelluminate,sharedsubpage,questionnaire,page,externalquiz,ouwiki,dualpane,repeatactivity,folder,htmlactivity,id_assessment,is_banked,score,assessment_type,due_date,weight,module_presentation_length,gender,region,highest_education,imd_band,age_band,num_of_prev_attempts,studied_credits,disability,final_result,date_registration,date_unregistration
0,6516,AAA,2014J,-23.0,0,3,23,2,0,0,0,0,0,0.0,0,0,0,0,0,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,269,M,Scotland,HE Qualification,80-90%,55<=,0,60,N,Pass,-52.0,NaN
1,6516,AAA,2014J,-22.0,33,13,34,0,0,2,0,0,0,0.0,0,0,0,0,0,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,269,M,Scotland,HE Qualification,80-90%,55<=,0,60,N,Pass,-52.0,NaN
2,6516,AAA,2014J,-20.0,13,12,8,1,0,7,0,0,0,0.0,0,0,0,0,0,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,269,M,Scotland,HE Qualification,80-90%,55<=,0,60,N,Pass,-52.0,NaN
3,6516,AAA,2014J,-17.0,0,2,0,3,2,0,0,0,0,0.0,0,0,0,0,0,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,269,M,Scotland,HE Qualification,80-90%,55<=,0,60,N,Pass,-52.0,NaN
4,6516,AAA,2014J,-12.0,1,1,0,0,0,0,0,0,0,0.0,0,0,0,0,0,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,269,M,Scotland,HE Qualification,80-90%,55<=,0,60,N,Pass,-52.0,NaN


In [4]:
df.columns

Index(['id_student', 'code_module', 'code_presentation', 'date', 'forumng',
       'homepage', 'oucontent', 'subpage', 'url', 'resource', 'glossary',
       'dataplus', 'oucollaborate', 'quiz', 'ouelluminate', 'sharedsubpage',
       'questionnaire', 'page', 'externalquiz', 'ouwiki', 'dualpane',
       'repeatactivity', 'folder', 'htmlactivity', 'id_assessment',
       'is_banked', 'score', 'assessment_type', 'due_date', 'weight',
       'module_presentation_length', 'gender', 'region', 'highest_education',
       'imd_band', 'age_band', 'num_of_prev_attempts', 'studied_credits',
       'disability', 'final_result', 'date_registration',
       'date_unregistration'],
      dtype='object')

In [5]:
# Create a "week" column relative to each code_presentation's starting date. 
# Dates 0.0 - 6.0 are week 1, 7.0 - 13.0 are week 2, etc. 
# The weeks *before* date 0.0 are assigned to negative values.

df['week'] = df.groupby('code_presentation')['date'].transform(
    lambda x: ((x // 7) + 1).where(x >= 0, x // 7)
)

In [6]:
# Add submission related features:
# - Relative submission date
# - Submission type (late vs. early)

df["relative_submission_date"] = df["due_date"] - df["date"]
df["submission_type"] = np.where(df["relative_submission_date"] < 0, "Late", "Early")




In [7]:
# Add feature that sums total VLE interactions

vle_columns = ['quiz', 'questionnaire', 'externalquiz', 'oucontent', 'page', 'resource', 'url', 'homepage', 
               'glossary', 'subpage', 'folder', 'forumng', 'oucollaborate', 'ouelluminate', 'ouwiki', 'sharedsubpage', 
               'dataplus', 'repeatactivity', 'dualpane', 'htmlactivity'
]

df['total_vle_interactions'] = df[vle_columns].sum(axis = 1)

In [ ]:
# Add VLE interaction category features:
# - Assessment type interaction (percentage)
# - Content type interaction (percentage)
# - Collaborative type interaction (percentage)

# This creates 3 metrics, each split further into 5 categories for a total of 15 columns: 
# - overall
# - pre-course
# - pre-course through week 1
# - pre-course through week 2
# - pre-course through week 3

student_totals = df.groupby(['id_student', 'code_presentation'])[vle_columns].sum()

assessment_types = ['quiz', 'questionnaire', 'externalquiz']
content_types = ['oucontent', 'page', 'resource', 'url', 'homepage', 'glossary', 'subpage', 'folder']
collaborative_types = ['forumng', 'oucollaborate', 'ouelluminate', 'ouwiki', 'sharedsubpage']

def calculate_focus_metrics(df_filtered, suffix):
    student_totals = df_filtered.groupby(['id_student', 'code_presentation'])[vle_columns].sum()
    
    assessment_focus = student_totals[assessment_types].sum(axis=1) / student_totals[vle_columns].sum(axis=1)
    assessment_focus = assessment_focus.fillna(0).rename(f"assessment_focus_{suffix}")
    
    content_focus = student_totals[content_types].sum(axis=1) / student_totals[vle_columns].sum(axis=1)
    content_focus = content_focus.fillna(0).rename(f"content_focus_{suffix}")
    
    collaborative_focus = student_totals[collaborative_types].sum(axis=1) / student_totals[vle_columns].sum(axis=1)
    collaborative_focus = collaborative_focus.fillna(0).rename(f"collaborative_focus_{suffix}")
    
    return assessment_focus, content_focus, collaborative_focus

# Pre-course only

df_pre = df[df['week'] < 0]
assessment_focus_pre, content_focus_pre, collaborative_focus_pre = calculate_focus_metrics(df_pre, 'pre')

# Pre-course through week 1

df_pre_w1 = df[df['week'] <= 1]
assessment_focus_pre_w1, content_focus_pre_w1, collaborative_focus_pre_w1 = calculate_focus_metrics(df_pre_w1, 'pre_w1')

# Pre-course through week 2

df_pre_w2 = df[df['week'] <= 2]
assessment_focus_pre_w2, content_focus_pre_w2, collaborative_focus_pre_w2 = calculate_focus_metrics(df_pre_w2, 'pre_w2')

# Pre-course through week 3

df_pre_w3 = df[df['week'] <= 3]
assessment_focus_pre_w3, content_focus_pre_w3, collaborative_focus_pre_w3 = calculate_focus_metrics(df_pre_w3, 'pre_w3')

for focus_metric in [assessment_focus_pre, content_focus_pre, collaborative_focus_pre,
                     assessment_focus_pre_w1, content_focus_pre_w1, collaborative_focus_pre_w1,
                     assessment_focus_pre_w2, content_focus_pre_w2, collaborative_focus_pre_w2,
                     assessment_focus_pre_w3, content_focus_pre_w3, collaborative_focus_pre_w3]:
    df = df.merge(focus_metric, on=['id_student', 'code_presentation'], how='left')

# Overall metrics

student_totals = df.groupby(['id_student', 'code_presentation'])[vle_columns].sum()
assessment_focus = student_totals[assessment_types].sum(axis=1) / student_totals[vle_columns].sum(axis=1)
assessment_focus = assessment_focus.fillna(0).rename("assessment_focus_overall")
content_focus = student_totals[content_types].sum(axis=1) / student_totals[vle_columns].sum(axis=1)
content_focus = content_focus.fillna(0).rename("content_focus_overall")
collaborative_focus = student_totals[collaborative_types].sum(axis=1) / student_totals[vle_columns].sum(axis=1)
collaborative_focus = collaborative_focus.fillna(0).rename("collaborative_focus_overall")
df = df.merge(assessment_focus, on=['id_student', 'code_presentation'], how='left')
df = df.merge(content_focus, on=['id_student', 'code_presentation'], how='left')
df = df.merge(collaborative_focus, on=['id_student', 'code_presentation'], how='left')


In [11]:
# Add regularity features:
# - Active days per week
# - Standard deviation of gaps between logins

# This creates 2 metrics, each split further into 5 categories for a total of 10 columns: 
# - overall
# - pre-course
# - pre-course through week 1
# - pre-course through week 2
# - pre-course through week 3

# Code to calculate a student's average login time per week
# This particular script might need to be modified to account for the pre-course data

def calculate_weekly_consistency(df_filtered, suffix, student_id_col='id_student', date_col='date', week_col='week'):
    weekly_days = df_filtered.groupby([student_id_col, week_col])[date_col].nunique()
    consistency = weekly_days.groupby(student_id_col).std()
    consistency = consistency.rename(f'active_days_per_week_{suffix}')
    return consistency

# Code to calculate standard deviation of gaps between logins
# This one also might need to be modified to account for the pre-course data

def calculate_regularity_std_period(df_filtered, suffix, student_id_col='id_student', date_col='date'):
    regularity = df_filtered.groupby(student_id_col)[date_col].apply(
        lambda x: x.sort_values().diff().std()
    )
    regularity = regularity.rename(f'std_regularity_{suffix}')
    return regularity

# 1. Pre-course only (negative weeks)
df_pre = df[df['week'] < 0]
active_days_pre = calculate_weekly_consistency(df_pre, 'pre')
std_regularity_pre = calculate_regularity_std_period(df_pre, 'pre')

# 2. Pre-course through week 1
df_pre_w1 = df[df['week'] <= 1]
active_days_pre_w1 = calculate_weekly_consistency(df_pre_w1, 'pre_w1')
std_regularity_pre_w1 = calculate_regularity_std_period(df_pre_w1, 'pre_w1')

# 3. Pre-course through week 2
df_pre_w2 = df[df['week'] <= 2]
active_days_pre_w2 = calculate_weekly_consistency(df_pre_w2, 'pre_w2')
std_regularity_pre_w2 = calculate_regularity_std_period(df_pre_w2, 'pre_w2')

# 4. Pre-course through week 3
df_pre_w3 = df[df['week'] <= 3]
active_days_pre_w3 = calculate_weekly_consistency(df_pre_w3, 'pre_w3')
std_regularity_pre_w3 = calculate_regularity_std_period(df_pre_w3, 'pre_w3')

# Merge all 8 new features back to original dataframe
for metric in [active_days_pre, std_regularity_pre,
               active_days_pre_w1, std_regularity_pre_w1,
               active_days_pre_w2, std_regularity_pre_w2,
               active_days_pre_w3, std_regularity_pre_w3]:
    df[metric.name] = df['id_student'].map(metric)

def weekly_consistency(df, student_id_col='id_student', date_col='date', week_col='week'):
    weekly_days = df.groupby([student_id_col, week_col])[date_col].nunique()
    consistency = weekly_days.groupby(student_id_col).std()
    df['active_days_per_week_overall'] = df[student_id_col].map(consistency)
    return df

df = weekly_consistency(df)

def calculate_regularity_std(df, student_id_col='id_student', date_col='date'):
    regularity = df.groupby(student_id_col)[date_col].apply(
        lambda x: x.sort_values().diff().std()
    )
    df['std_regularity_overall'] = df[student_id_col].map(regularity)
    return df

df = calculate_regularity_std(df)



In [ ]:
# Add diversity of interaction features
# - VLE richness (number of different VLE types used) 
# - Shannon entropy (overall diversity of interactions)

from scipy.stats import entropy

# Calculates Shannon entropy

def shannon_entropy_calc(counts, vle_columns):
    counts = counts[vle_columns].values.astype(float)
    counts = counts[counts > 0]
    if len(counts) == 0:
        return 0
    proportions = counts / counts.sum()
    return entropy(proportions, base=2)

# Calculates VLE metrics for a given time period

def calculate_vle_metrics(df_filtered, suffix, vle_columns):
    # Group by student and sum VLE activities for the period
    student_vle = df_filtered.groupby('id_student')[vle_columns].sum()
    
    # Calculate richness (number of VLE types used)
    richness = (student_vle > 0).sum(axis=1).rename(f'vle_richness_{suffix}')
    
    # Calculate Shannon entropy
    diversity = student_vle.apply(lambda row: shannon_entropy_calc(row, vle_columns), axis=1).rename(f'diversity_shannon_{suffix}')
    
    return richness, diversity

# 1. Pre-course only (negative weeks)
df_pre = df[df['week'] < 0]
vle_richness_pre, diversity_shannon_pre = calculate_vle_metrics(df_pre, 'pre', vle_columns)

# 2. Pre-course through week 1
df_pre_w1 = df[df['week'] <= 1]
vle_richness_pre_w1, diversity_shannon_pre_w1 = calculate_vle_metrics(df_pre_w1, 'pre_w1', vle_columns)

# 3. Pre-course through week 2
df_pre_w2 = df[df['week'] <= 2]
vle_richness_pre_w2, diversity_shannon_pre_w2 = calculate_vle_metrics(df_pre_w2, 'pre_w2', vle_columns)

# 4. Pre-course through week 3
df_pre_w3 = df[df['week'] <= 3]
vle_richness_pre_w3, diversity_shannon_pre_w3 = calculate_vle_metrics(df_pre_w3, 'pre_w3', vle_columns)

# Merge all 8 new features back to original dataframe
for metric in [vle_richness_pre, diversity_shannon_pre,
               vle_richness_pre_w1, diversity_shannon_pre_w1,
               vle_richness_pre_w2, diversity_shannon_pre_w2,
               vle_richness_pre_w3, diversity_shannon_pre_w3]:
    df[metric.name] = df['id_student'].map(metric)

# All data - calculated per row
df['vle_richness'] = (df[vle_columns] > 0).sum(axis=1)

def shannon_entropy(row):
    counts = row[vle_columns].values.astype(float)
    counts = counts[counts > 0]
    if len(counts) == 0:
        return 0
    proportions = counts / counts.sum()
    return entropy(proportions, base=2)

df['diversity_shannon'] = df.apply(shannon_entropy, axis=1)

In [14]:
df.columns

Index(['id_student', 'code_module', 'code_presentation', 'date', 'forumng',
       'homepage', 'oucontent', 'subpage', 'url', 'resource', 'glossary',
       'dataplus', 'oucollaborate', 'quiz', 'ouelluminate', 'sharedsubpage',
       'questionnaire', 'page', 'externalquiz', 'ouwiki', 'dualpane',
       'repeatactivity', 'folder', 'htmlactivity', 'id_assessment',
       'is_banked', 'score', 'assessment_type', 'due_date', 'weight',
       'module_presentation_length', 'gender', 'region', 'highest_education',
       'imd_band', 'age_band', 'num_of_prev_attempts', 'studied_credits',
       'disability', 'final_result', 'date_registration',
       'date_unregistration', 'week', 'relative_submission_date',
       'submission_type', 'total_vle_interactions', 'assessment_focus_pre',
       'content_focus_pre', 'collaborative_focus_pre',
       'assessment_focus_pre_w1', 'content_focus_pre_w1',
       'collaborative_focus_pre_w1', 'assessment_focus_pre_w2',
       'content_focus_pre_w2', 'coll

In [15]:
df.head(10)

,id_student,code_module,code_presentation,date,forumng,homepage,oucontent,subpage,url,resource,glossary,dataplus,oucollaborate,quiz,ouelluminate,sharedsubpage,questionnaire,page,externalquiz,ouwiki,dualpane,repeatactivity,folder,htmlactivity,id_assessment,...,content_focus_pre_w3,collaborative_focus_pre_w3,assessment_focus_overall,content_focus_overall,collaborative_focus_overall,active_days_per_week_pre,std_regularity_pre,active_days_per_week_pre_w1,std_regularity_pre_w1,active_days_per_week_pre_w2,std_regularity_pre_w2,active_days_per_week_pre_w3,std_regularity_pre_w3,active_days_per_week_overall,std_regularity_overall,vle_richness_pre,diversity_shannon_pre,vle_richness_pre_w1,diversity_shannon_pre_w1,vle_richness_pre_w2,diversity_shannon_pre_w2,vle_richness_pre_w3,diversity_shannon_pre_w3,vle_richness,diversity_shannon
0,6516,AAA,2014J,-23.0,0,3,23,2,0,0,0,0,0,0.0,0,0,0,0,0,0,0,0,0,0,NaN,...,0.739274,0.260726,0.0,0.830885,0.161591,1.258306,1.908627,2.0,1.63915,1.834848,1.49509,1.812654,1.358621,1.628442,2.439529,6.0,2.00361,6.0,2.09697,6.0,2.129854,6.0,2.094273,3,0.850326
1,6516,AAA,2014J,-22.0,33,13,34,0,0,2,0,0,0,0.0,0,0,0,0,0,0,0,0,0,0,NaN,...,0.739274,0.260726,0.0,0.830885,0.161591,1.258306,1.908627,2.0,1.63915,1.834848,1.49509,1.812654,1.358621,1.628442,2.439529,6.0,2.00361,6.0,2.09697,6.0,2.129854,6.0,2.094273,4,1.607010
2,6516,AAA,2014J,-20.0,13,12,8,1,0,7,0,0,0,0.0,0,0,0,0,0,0,0,0,0,0,NaN,...,0.739274,0.260726,0.0,0.830885,0.161591,1.258306,1.908627,2.0,1.63915,1.834848,1.49509,1.812654,1.358621,1.628442,2.439529,6.0,2.00361,6.0,2.09697,6.0,2.129854,6.0,2.094273,5,2.070314
3,6516,AAA,2014J,-17.0,0,2,0,3,2,0,0,0,0,0.0,0,0,0,0,0,0,0,0,0,0,NaN,...,0.739274,0.260726,0.0,0.830885,0.161591,1.258306,1.908627,2.0,1.63915,1.834848,1.49509,1.812654,1.358621,1.628442,2.439529,6.0,2.00361,6.0,2.09697,6.0,2.129854,6.0,2.094273,3,1.556657
4,6516,AAA,2014J,-12.0,1,1,0,0,0,0,0,0,0,0.0,0,0,0,0,0,0,0,0,0,0,NaN,...,0.739274,0.260726,0.0,0.830885,0.161591,1.258306,1.908627,2.0,1.63915,1.834848,1.49509,1.812654,1.358621,1.628442,2.439529,6.0,2.00361,6.0,2.09697,6.0,2.129854,6.0,2.094273,2,1.000000
5,6516,AAA,2014J,-6.0,12,4,0,0,0,0,0,0,0,0.0,0,0,0,0,0,0,0,0,0,0,NaN,...,0.739274,0.260726,0.0,0.830885,0.161591,1.258306,1.908627,2.0,1.63915,1.834848,1.49509,1.812654,1.358621,1.628442,2.439529,6.0,2.00361,6.0,2.09697,6.0,2.129854,6.0,2.094273,2,0.811278
6,6516,AAA,2014J,-5.0,0,7,5,1,0,2,0,0,0,0.0,0,0,0,0,0,0,0,0,0,0,NaN,...,0.739274,0.260726,0.0,0.830885,0.161591,1.258306,1.908627,2.0,1.63915,1.834848,1.49509,1.812654,1.358621,1.628442,2.439529,6.0,2.00361,6.0,2.09697,6.0,2.129854,6.0,2.094273,4,1.689482
7,6516,AAA,2014J,-2.0,2,8,23,5,2,0,0,0,0,0.0,0,0,0,0,0,0,0,0,0,0,NaN,...,0.739274,0.260726,0.0,0.830885,0.161591,1.258306,1.908627,2.0,1.63915,1.834848,1.49509,1.812654,1.358621,1.628442,2.439529,6.0,2.00361,6.0,2.09697,6.0,2.129854,6.0,2.094273,5,1.730639
8,6516,AAA,2014J,-1.0,3,7,15,0,0,0,0,0,0,0.0,0,0,0,0,0,0,0,0,0,0,NaN,...,0.739274,0.260726,0.0,0.830885,0.161591,1.258306,1.908627,2.0,1.63915,1.834848,1.49509,1.812654,1.358621,1.628442,2.439529,6.0,2.00361,6.0,2.09697,6.0,2.129854,6.0,2.094273,3,1.323467
9,6516,AAA,2014J,0.0,48,11,1,5,6,0,0,0,0,0.0,0,0,0,0,0,0,0,0,0,0,NaN,...,0.739274,0.260726,0.0,0.830885,0.161591,1.258306,1.908627,2.0,1.63915,1.834848,1.49509,1.812654,1.358621,1.628442,2.439529,6.0,2.00361,6.0,2.09697,6.0,2.129854,6.0,2.094273,5,1.456066


In [16]:
#df.loc[df["id_assessment"].notna(), [
#        "id_student", "date", "id_assessment", "score",
#    "assessment_type", "due_date", "relative_submission_date", "submission_type"
#]].head()

,id_student,date,id_assessment,score,assessment_type,due_date,relative_submission_date,submission_type
21,6516,17.0,1758.0,60.0,TMA,19.0,2.0,Early
40,6516,51.0,1759.0,48.0,TMA,54.0,3.0,Early
73,6516,116.0,1760.0,63.0,TMA,117.0,1.0,Early
103,6516,164.0,1761.0,61.0,TMA,166.0,2.0,Early
136,6516,210.0,1762.0,77.0,TMA,215.0,5.0,Early


Separate modules and save to csv

In [17]:
# Create a folder to store the CSV files (optional but recommended)
output_folder = "modules_csv"
os.makedirs(output_folder, exist_ok=True)

# Loop over each unique code_module and save separately
for module, df_subset in df.groupby("code_module"):
    filename = f"{output_folder}/{module}_data.csv"
    df_subset.to_csv(filename, index=False)
    print(f"Saved: {filename}")

Saved: modules_csv/AAA_data.csv
Saved: modules_csv/BBB_data.csv
Saved: modules_csv/CCC_data.csv
Saved: modules_csv/DDD_data.csv
Saved: modules_csv/EEE_data.csv
Saved: modules_csv/FFF_data.csv
Saved: modules_csv/GGG_data.csv
